# 📸 debim: 407-Model 100% Exhaustive 3D Visual Regression Suite

[![GitHub Repository](https://img.shields.io/badge/GitHub-PRIDA--TAKON%2Fdebim-181717?logo=github&style=for-the-badge)](https://github.com/PRIDA-TAKON/debim)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg?style=for-the-badge)](https://github.com/PRIDA-TAKON/debim/blob/main/LICENSE)

> **debim** (Declarative BIM / Building-as-Code) Global 3D Visual Regression Suite.
> Evaluates **100% of all components** across the entire 407-model open benchmark using 4 parallel CPU workers.
>
> Captures 3D Bounding Box IoU (%) and Pixel Diff Heatmap (%) for every physical component to ensure declarative fidelity.

In [ ]:
# 1. Install dependencies & clone latest debim repository
!pip install -q ifcopenshell "pydantic>=2.0" pyyaml rich trimesh matplotlib pillow pandas tabulate

!rm -rf /kaggle/working/debim
!git clone https://github.com/PRIDA-TAKON/debim.git /kaggle/working/debim

import sys
sys.path.insert(0, "/kaggle/working/debim/src")
sys.path.insert(0, "/kaggle/working/debim")
print("Environment initialized successfully.")

In [ ]:
# 2. Discover and index all unique real-world IFC models
from pathlib import Path
import pandas as pd

input_root = Path("/kaggle/input")
all_ifc_files = sorted(list(input_root.glob("**/*.ifc")))
print(f"Discovered {len(all_ifc_files)} raw IFC files in input datasets.")

# Deduplicate by (filename, file_size)
unique_models = []
seen_signatures = set()
for f in all_ifc_files:
    size = f.stat().st_size
    if size < 5000: # Skip empty or micro snippets (<5KB)
        continue
    sig = (f.name, size)
    if sig not in seen_signatures:
        seen_signatures.add(sig)
        unique_models.append(f)

print(f"Indexed {len(unique_models)} unique benchmark models for 100% exhaustive 3D testing.")

In [ ]:
# 3. Parallel 4-Core 3D Visual Regression Engine
import concurrent.futures
import os
import time
from pathlib import Path
from tools.render_visual_regression import run_visual_regression

output_root = Path("/kaggle/working/visual_regression_output")
output_root.mkdir(parents=True, exist_ok=True)

def worker_task(ifc_path_str):
    ifc_p = Path(ifc_path_str)
    model_dir = output_root / ifc_p.stem
    try:
        stat = run_visual_regression(
            ifc_path=ifc_p,
            output_dir=model_dir,
            max_elements_per_type=99999,
            limit_total=99999,
            save_images=True,
            save_passes_sample_limit=1
        )
        return {"model_name": ifc_p.name, "size_kb": round(ifc_p.stat().st_size / 1024, 1), **stat}
    except Exception as e:
        return {
            "model_name": ifc_p.name,
            "size_kb": round(ifc_p.stat().st_size / 1024, 1),
            "total_tested": 0,
            "pass_count": 0,
            "avg_match": 0.0,
            "avg_iou": 0.0,
            "report_path": None,
            "duration": 0.0,
            "status": "FATAL_ERROR",
            "error": str(e)
        }

print(f"Launching 4-worker parallel execution across {len(unique_models)} models...")
t_start = time.time()
results = []
completed_count = 0

with concurrent.futures.ProcessPoolExecutor(max_workers=4) as executor:
    future_to_model = {executor.submit(worker_task, str(m)): m for m in unique_models}
    for future in concurrent.futures.as_completed(future_to_model):
        completed_count += 1
        res = future.result()
        results.append(res)
        if completed_count % 10 == 0 or completed_count == len(unique_models):
            elapsed = time.time() - t_start
            print(f"[{completed_count}/{len(unique_models)}] Elapsed: {elapsed/60:.1f}m | Last: {res['model_name']} (Match: {res.get('avg_match', 0):.1f}%, Tested: {res.get('total_tested', 0)})")

print(f"All {len(unique_models)} models processed in {(time.time() - t_start)/60:.1f} minutes.")

In [ ]:
# 4. Generate Master Leaderboard & Visual KPI Analytics
import pandas as pd
from IPython.display import display, Markdown, Image

df = pd.DataFrame(results)
leaderboard_path = Path("/kaggle/working/visual_regression_leaderboard.csv")
df.to_csv(leaderboard_path, index=False)
print(f"Saved master leaderboard to: {leaderboard_path}")

# Filter models that had tested elements
tested_df = df[df["total_tested"] > 0]
total_pieces = int(tested_df["total_tested"].sum()) if len(tested_df) > 0 else 0
total_passed = int(tested_df["pass_count"].sum()) if len(tested_df) > 0 else 0
global_pass_rate = (total_passed / total_pieces * 100) if total_pieces > 0 else 0.0
mean_pixel_match = float(tested_df["avg_match"].mean()) if len(tested_df) > 0 else 0.0
mean_iou = float(tested_df["avg_iou"].mean()) if len(tested_df) > 0 else 0.0

display(Markdown(f"""
# 🏆 Global 3D Visual Regression Benchmark Results
- **Total Models Evaluated:** {len(df)} models
- **Models with Valid 3D Components:** {len(tested_df)} models
- **Total 3D Components Tested (100% Coverage):** {total_pieces:,} pieces
- **Global Pass Rate (>= 85% match):** {total_passed:,}/{total_pieces:,} ({global_pass_rate:.1f}%)
- **Mean Visual Match Score:** {mean_pixel_match:.1f}%
- **Mean 3D Bounding Box IoU:** {mean_iou:.1f}%
"""))

display(tested_df.sort_values(by="avg_match", ascending=False).head(30))

# Visual Diff Inspection of Notable Components
diff_images = list(output_root.glob("**/*.png"))
print(f"\nCaptured {len(diff_images)} visual diff inspection panels.")

display(Markdown("### 🔍 Representative 3D Visual Diffs (Original | debim | Diff Map)"))
for img in diff_images[:16]:
    display(Markdown(f"**{img.parent.parent.name} - {img.stem}**"))
    display(Image(filename=str(img)))

# 5. Clean up repository clone to minimize output artifact download size
!rm -rf /kaggle/working/debim
print("Cleanup complete. Output folder contains only benchmark reports, CSV, and visual diffs.")